# Ridge Regression

Learn this model in order: **setup → baseline → initial model → tuning → interpretation**.

We keep all 395 students, including 38 recorded zero grades. The same 316 training students and five folds are used in every notebook. The 79 test students belong to final evaluation in Notebook 04.

There are two clearly separated scenarios: without G1/G2 (main) and with both grades (later stage). Input timing, particularly absences, still needs verification. MAE remains the selection metric. The test results have already been inspected: rerunning the unchanged experiment reproduces it, rather than creating a new unseen test.

Results and explanations stay in this notebook. Only fitted pipelines are saved to `models/` for the comparison and evaluation notebooks.

## 1 Setup

Shared helpers in `src/model_utils.py` handle the split and repeated metric calculations. Model settings, fitting and tuning remain visible here. Preprocessing is always fitted inside the training fold.

MAE/RMSE measure grade-point error; R² describes squared-error performance. Accuracy, precision, recall, F1 and specificity evaluate the derived **predicted G3 < 10** flag against **actual G3 < 10**. No rounding or threshold tuning is used.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


In [2]:
MODEL_NAME = 'Ridge Regression'
MODEL_SLUG = 'ridge'
SCALE_NUMERIC = True

from sklearn.linear_model import Ridge
INITIAL_ESTIMATOR = Ridge(alpha=1.0)
MODEL_PARAM_GRID = {'model__alpha': [0.01, 0.1, 1, 10, 100, 1000]}

## 2 Constant baselines

Mean and median baselines learn their constant from each fold's training labels. They ignore the inputs, so one set of scores applies to both scenarios. Precision is undefined when nobody is flagged; missing values are left visible rather than replaced with perfect scores.

In [3]:
baseline_folds = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(DummyRegressor(strategy=strategy), X_no_grades, y_train,
        cv=CV, scoring=SCORING, error_score="raise")
    baseline_folds[strategy] = fold_results(scores)

display(pd.DataFrame({name: folds[METRICS].mean()
    for name, folds in baseline_folds.items()}).T.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4029,4.5635,-0.0350,0.6744,NaN,0.0,0.0,1.0
median,3.3403,4.5931,-0.0519,0.6744,NaN,0.0,0.0,1.0


## 3 Without G1 and G2

This is the main prediction scenario. Excluding grades does not by itself prove that all other inputs are available early.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [4]:
initial_no_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=False)
initial_scores_no_grades = cross_validate(initial_no_grades, X_no_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_no_grades = fold_results(initial_scores_no_grades)
display(cv_summary(initial_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4904,4.5621,-0.0694,0.6422,0.4668,0.6023,0.5178,0.6673
std,0.4214,0.3529,0.2330,0.0430,0.1326,0.0785,0.0877,0.0722
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [5]:
grid_no_grades = dict(MODEL_PARAM_GRID)
grid_no_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=False)
    for use_log in [False, True]
]
search_no_grades = GridSearchCV(initial_no_grades, grid_no_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_no_grades.fit(X_no_grades, y_train)
best_no_grades = search_no_grades.best_estimator_
tuned_folds_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_)

print("Model settings:", best_no_grades.named_steps["model"])
print("Log absences:", best_no_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.2423,4.1677,0.1253,0.6552,0.4695,0.5284,0.4890,0.7192
std,0.4454,0.4779,0.1012,0.0322,0.0864,0.1466,0.0853,0.0490
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: Ridge(alpha=100)
Log absences: True


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [6]:
comparison_no_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_no_grades[METRICS].mean(),
    "Tuned model": tuned_folds_no_grades[METRICS].mean(),
}).T
display(comparison_no_grades.round(4))

training_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_no_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_no_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_no_grades),
    "Tuned model": pooled_risk(tuned_folds_no_grades)}).T.round(4))
display(tuned_folds_no_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,3.4904,4.5621,-0.0694,0.6422,0.4668,0.6023,0.5178,0.6673
Tuned model,3.2423,4.1677,0.1253,0.6552,0.4695,0.5284,0.4890,0.7192


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.6424,0.4627,0.6019,0.5232,0.6620,62.0,72.0,41.0,141.0
Tuned model,0.6551,0.4737,0.5243,0.4977,0.7183,54.0,60.0,49.0,153.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,2.7806,3.7474,0.0220,0.6094,0.5714,0.4286,0.4898,0.7500,27.0,9.0,16.0,12.0
1,2,3.7838,4.7317,0.2565,0.6825,0.5161,0.7619,0.6154,0.6429,27.0,15.0,5.0,16.0
2,3,2.7917,3.6003,0.0771,0.6349,0.3684,0.3889,0.3784,0.7333,33.0,12.0,11.0,7.0
3,4,3.5156,4.3203,0.0626,0.6667,0.3913,0.5625,0.4615,0.7021,33.0,14.0,7.0,9.0
4,5,3.3395,4.4387,0.2081,0.6825,0.5000,0.5000,0.5000,0.7674,33.0,10.0,10.0,10.0


Tuned training MAE: 2.9836
Tuned validation MAE: 3.2423


## 4 With G1 and G2

This is a later-stage comparison, usable only once both prior grades are known. It answers a different prediction-timing question.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [7]:
initial_with_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=True)
initial_scores_with_grades = cross_validate(initial_with_grades, X_with_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_with_grades = fold_results(initial_scores_with_grades)
display(cv_summary(initial_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.3705,1.9638,0.8080,0.8325,0.6774,0.9424,0.7858,0.7756
std,0.2529,0.3985,0.0369,0.0749,0.1086,0.0685,0.0902,0.1016
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [8]:
grid_with_grades = dict(MODEL_PARAM_GRID)
grid_with_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=True)
    for use_log in [False, True]
]
search_with_grades = GridSearchCV(initial_with_grades, grid_with_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_with_grades.fit(X_with_grades, y_train)
best_with_grades = search_with_grades.best_estimator_
tuned_folds_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_)

print("Model settings:", best_with_grades.named_steps["model"])
print("Log absences:", best_with_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.2955,1.8116,0.8358,0.8515,0.7070,0.9495,0.8087,0.7964
std,0.2068,0.3440,0.0304,0.0739,0.0982,0.0686,0.0821,0.1172
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: Ridge(alpha=10)
Log absences: True


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [9]:
comparison_with_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_with_grades[METRICS].mean(),
    "Tuned model": tuned_folds_with_grades[METRICS].mean(),
}).T
display(comparison_with_grades.round(4))

training_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_with_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_with_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_with_grades),
    "Tuned model": pooled_risk(tuned_folds_with_grades)}).T.round(4))
display(tuned_folds_with_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,1.3705,1.9638,0.8080,0.8325,0.6774,0.9424,0.7858,0.7756
Tuned model,1.2955,1.8116,0.8358,0.8515,0.7070,0.9495,0.8087,0.7964


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.8323,0.6736,0.9417,0.7854,0.7793,97.0,47.0,6.0,166.0
Tuned model,0.8513,0.7000,0.9515,0.8066,0.8028,98.0,42.0,5.0,171.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,1.1907,1.5647,0.8295,0.7656,0.6585,0.9643,0.7826,0.6111,22.0,14.0,1.0,27.0
1,2,1.5510,2.2508,0.8318,0.9048,0.7778,1.0000,0.8750,0.8571,36.0,6.0,0.0,21.0
2,3,1.2056,1.6375,0.8091,0.7778,0.5769,0.8333,0.6818,0.7556,34.0,11.0,3.0,15.0
3,4,1.0591,1.4950,0.8878,0.8889,0.6957,1.0000,0.8205,0.8511,40.0,7.0,0.0,16.0
4,5,1.4711,2.1099,0.8211,0.9206,0.8261,0.9500,0.8837,0.9070,39.0,4.0,1.0,19.0


Tuned training MAE: 1.1119
Tuned validation MAE: 1.2955


## 5 Compare scenarios and save the fitted models

The score difference reflects the information available at different prediction times. Do not present the G1/G2 scenario as an early prediction. Save only the two complete fitted pipelines; tables and findings remain inside the notebook.

In [10]:
display(pd.DataFrame({
    "Without G1/G2": tuned_folds_no_grades[METRICS].mean(),
    "With G1/G2": tuned_folds_with_grades[METRICS].mean(),
}).T.round(4))

model_dir = PROJECT_ROOT / "models/experiments"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_no_grades, model_dir / "ridge__no_prior_grades.joblib")
joblib.dump(best_with_grades, model_dir / "ridge__with_prior_grades.joblib")
print("Saved fitted pipelines. No experiment tables or reports are exported.")

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Without G1/G2,3.2423,4.1677,0.1253,0.6552,0.4695,0.5284,0.4890,0.7192
With G1/G2,1.2955,1.8116,0.8358,0.8515,0.7070,0.9495,0.8087,0.7964


Saved fitted pipelines. No experiment tables or reports are exported.
